# Case B — Fintech (`refresh_account_features`)
Walks the M3/M4 path: transactions, MVCC snapshots, online feature view (point-in-time lookup).

In [ ]:
from pathlib import Path
import tempfile, pyarrow as pa
from caracaldb.storage import create_bundle
from caracaldb.storage.node_store import open_node_store
from caracaldb.storage.snapshot import create_snapshot
from caracaldb.storage.wal import Wal
from caracaldb.tx import TransactionManager
from caracaldb.feature import OnlineFeatureView

tmp = Path(tempfile.mkdtemp())
bundle = create_bundle(tmp / 'fintech')
accts = open_node_store(bundle, class_iri='http://x/Account', local_name='Account', create=True)
accts.append(pa.record_batch({'name': pa.array(['a0','a1','a2','a3','a4']),
                              'balance': pa.array([100.0,200.0,300.0,400.0,500.0])}))
snap = create_snapshot(bundle, 'v1')
wal = Wal(bundle.path / 'wal')
mgr = TransactionManager(wal)
with mgr.transaction() as tx:
    tx.record_write('Account', 1)
print('snapshot:', snap, 'committed_lsn:', tx.committed_lsn)


In [ ]:
view = OnlineFeatureView(bundle, class_iri='http://x/Account', local_name='Account',
                          feature_columns=['balance', 'name'])
print('lookup(2):', view.lookup(2))
print('stats:', view.stats())
